In [1]:
# %% [markdown]
# # Notebook 03 — Intégration IPFS
#
# **Objectif** : Valider les opérations IPFS du système
# (upload, retrieve, vérification d'intégrité).
#
# **Prérequis** :
# - Daemon IPFS démarré : `ipfs daemon`
# - Port API IPFS disponible sur `http://127.0.0.1:5001`

# %%
import sys, os, json, hashlib, time
import numpy as np

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from backend.core.ipfs_storage import IPFSStorage
from backend.config            import settings

ipfs = IPFSStorage()
print(f"API IPFS     : {ipfs.api_url}")
print(f"Gateway IPFS : {ipfs.gateway_url}")

# %% [markdown]
# ## 1. Vérification de la disponibilité du daemon

# %%
available = ipfs.is_available()
print(f"Daemon IPFS disponible : {'✅ OUI' if available else '❌ NON'}")

if not available:
    print("\n⚠️  Démarrer le daemon IPFS avant de continuer :")
    print("   ipfs daemon")
    raise SystemExit("Daemon IPFS requis")


API IPFS     : http://127.0.0.1:5001
Gateway IPFS : http://127.0.0.1:8080
Daemon IPFS disponible : ✅ OUI


In [2]:
# %% [markdown]
# ## 2. Upload de fichiers

# %%
# Test 1 : Upload d'un fichier texte simple
print("Test 1 : Upload fichier texte")
print("─" * 40)

test_content = (
    b"Rapport forensique de test\n"
    b"Oeuvre : Ma Chanson\n"
    b"Score  : 0.9742\n"
    b"Hash   : abc123def456\n"
)

t0  = time.time()
cid = ipfs.upload_file(test_content, filename="test_rapport.txt")
dt  = time.time() - t0

print(f"  CID      : {cid}")
print(f"  Taille   : {len(test_content)} bytes")
print(f"  Temps    : {dt:.3f}s")
assert cid.startswith("Qm") or cid.startswith("bafy"), \
    f"CID IPFS invalide : {cid}"
print("  ✅ Upload texte validé")

# %%
# Test 2 : Upload d'un fichier JSON (rapport forensique)
print("\nTest 2 : Upload rapport forensique JSON")
print("─" * 40)

rapport = {
    "proof_id":      "test-uuid-001",
    "generated_at":  "2025-05-07T10:00:00",
    "match_score":   0.9742,
    "threshold":     0.85,
    "original_work": "a" * 64,
    "suspect_hash":  "b" * 64,
    "suspect_cid":   "QmSuspect123",
    "original_cid":  "QmOriginal456",
    "confidence":    "TRÈS HAUTE"
}

t0        = time.time()
cid_json  = ipfs.upload_json(rapport, filename="rapport_forensique.json")
dt        = time.time() - t0

print(f"  CID JSON : {cid_json}")
print(f"  Temps    : {dt:.3f}s")
assert cid_json != cid, "Deux contenus différents doivent avoir des CID différents"
print("  ✅ Upload JSON validé")

# %%
# Test 3 : Upload d'un fichier audio synthétique
import soundfile as sf, tempfile

print("\nTest 3 : Upload fichier audio WAV")
print("─" * 40)

sr     = 8000
signal = np.sin(2 * np.pi * 440 * np.linspace(0, 3.0, sr * 3)).astype(np.float32)
tmp    = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
sf.write(tmp.name, signal, sr)
tmp.close()   # Fermer explicitement le handle

# Maintenant, ouvrir en lecture binaire et fermer juste après
with open(tmp.name, "rb") as f:
    audio_bytes = f.read()

t0        = time.time()
cid_audio = ipfs.upload_file(audio_bytes, filename="test_audio.wav")
dt        = time.time() - t0

# Tentative de suppression, sans bloquer si impossible
try:
    os.unlink(tmp.name)
except PermissionError:
    print(f"  (Fichier temporaire non supprimé, sera nettoyé plus tard)")

print(f"  CID audio : {cid_audio}")
print(f"  Taille    : {len(audio_bytes):,} bytes")
print(f"  Temps     : {dt:.3f}s")
print("  ✅ Upload audio validé")

Test 1 : Upload fichier texte
────────────────────────────────────────
  CID      : QmajWL6tEHaw7sSaL5VaNqDjgm19qofKaV7Xfsiy7N9zLf
  Taille   : 85 bytes
  Temps    : 0.100s
  ✅ Upload texte validé

Test 2 : Upload rapport forensique JSON
────────────────────────────────────────
  CID JSON : QmRLQVBf4b6scRw435zQdJXbRFn8wgj9gKTgikjTdwMuFW
  Temps    : 0.068s
  ✅ Upload JSON validé

Test 3 : Upload fichier audio WAV
────────────────────────────────────────
  CID audio : QmbZRF7iYNUMoQYoes51QZZ9CC5VQYhs8j9UmkwHhFaeXq
  Taille    : 48,044 bytes
  Temps     : 0.071s
  ✅ Upload audio validé


In [3]:
# %% [markdown]
# ## 3. Récupération de fichiers

# %%
print("Test récupération — texte")
print("─" * 40)

retrieved = ipfs.retrieve_file(cid)

assert retrieved == test_content, \
    f"Contenu récupéré différent du contenu original !"
print(f"  Contenu original  : {len(test_content)} bytes")
print(f"  Contenu récupéré  : {len(retrieved)} bytes")
print(f"  Correspondance    : ✅ IDENTIQUES")

# %%
print("\nTest récupération — JSON")
print("─" * 40)

retrieved_json_bytes = ipfs.retrieve_file(cid_json)
retrieved_json       = json.loads(retrieved_json_bytes.decode("utf-8"))

assert retrieved_json["proof_id"]    == rapport["proof_id"],    "proof_id incorrect"
assert retrieved_json["match_score"] == rapport["match_score"], "match_score incorrect"
print(f"  proof_id    : {retrieved_json['proof_id']}  ✅")
print(f"  match_score : {retrieved_json['match_score']}  ✅")
print("  ✅ Récupération JSON validée")

Test récupération — texte
────────────────────────────────────────
  Contenu original  : 85 bytes
  Contenu récupéré  : 85 bytes
  Correspondance    : ✅ IDENTIQUES

Test récupération — JSON
────────────────────────────────────────
  proof_id    : test-uuid-001  ✅
  match_score : 0.9742  ✅
  ✅ Récupération JSON validée


In [4]:
# %% [markdown]
# ## 4. Vérification d'intégrité

# %%
print("Test vérification d'intégrité")
print("─" * 40)

# Hash attendu du contenu texte
expected_hash = hashlib.sha256(test_content).hexdigest()
print(f"  Hash SHA-256 attendu : {expected_hash[:32]}...")

# Vérification avec le bon hash
result_ok = ipfs.verify_integrity(cid, expected_hash)
print(f"  Intégrité (bon hash) : {'✅ INTÈGRE' if result_ok else '❌ COMPROMIS'}")
assert result_ok, "L'intégrité devrait être validée avec le bon hash"

# Vérification avec un hash erroné
wrong_hash    = "0" * 64
result_wrong  = ipfs.verify_integrity(cid, wrong_hash)
print(f"  Intégrité (mauvais hash) : {'⚠️  ALTÉRÉ' if not result_wrong else '❌ Devrait détecter l altération'}")
assert not result_wrong, "L'intégrité devrait échouer avec un mauvais hash"

# Vérification sans hash (simple accessibilité)
result_simple = ipfs.verify_integrity(cid)
print(f"  Accessibilité simple : {'✅ ACCESSIBLE' if result_simple else '❌ INACCESSIBLE'}")
assert result_simple

print("  ✅ Toutes les vérifications d'intégrité validées")

Intégrité compromise pour QmajWL6tEHaw7sSaL5VaNqDjgm19qofKaV7Xfsiy7N9zLf : attendu 0000000000000000..., obtenu  f34f392649f2b9a9...


Test vérification d'intégrité
────────────────────────────────────────
  Hash SHA-256 attendu : f34f392649f2b9a9044e6695bfd24ea7...
  Intégrité (bon hash) : ✅ INTÈGRE
  Intégrité (mauvais hash) : ⚠️  ALTÉRÉ
  Accessibilité simple : ✅ ACCESSIBLE
  ✅ Toutes les vérifications d'intégrité validées


In [5]:
# %% [markdown]
# ## 5. Test CID inexistant

# %%
print("Test CID inexistant")
print("─" * 40)

fake_cid    = "QmFakeCIDThatDoesNotExist1234567890abcdef"
result_fake = ipfs.verify_integrity(fake_cid)
print(f"  CID inexistant → accessibilité : {'❌ INACCESSIBLE (attendu)' if not result_fake else '⚠️  Devrait être inaccessible'}")


Test CID inexistant
────────────────────────────────────────
  CID inexistant → accessibilité : ❌ INACCESSIBLE (attendu)


In [6]:
# %% [markdown]
# ## 6. URL Gateway

# %%
print("Test URL Gateway")
print("─" * 40)

for label, c in [("texte", cid), ("JSON", cid_json), ("audio", cid_audio)]:
    url = ipfs.get_gateway_url(c)
    print(f"  {label:6s} : {url}")

Test URL Gateway
────────────────────────────────────────
  texte  : http://127.0.0.1:8080/ipfs/QmajWL6tEHaw7sSaL5VaNqDjgm19qofKaV7Xfsiy7N9zLf
  JSON   : http://127.0.0.1:8080/ipfs/QmRLQVBf4b6scRw435zQdJXbRFn8wgj9gKTgikjTdwMuFW
  audio  : http://127.0.0.1:8080/ipfs/QmbZRF7iYNUMoQYoes51QZZ9CC5VQYhs8j9UmkwHhFaeXq


In [7]:
# %% [markdown]
# ## 7. Benchmark upload/download

# %%
print("\nBenchmark upload/download")
print("─" * 40)

sizes    = [1_000, 10_000, 100_000, 500_000]
up_times = []
dl_times = []

for size in sizes:
    data = os.urandom(size)

    t0 = time.time()
    c  = ipfs.upload_file(data, filename=f"bench_{size}.bin")
    up_times.append(time.time() - t0)

    t0 = time.time()
    ipfs.retrieve_file(c)
    dl_times.append(time.time() - t0)

    print(f"  {size/1000:6.0f} KB — "
          f"upload : {up_times[-1]*1000:.1f}ms  "
          f"download : {dl_times[-1]*1000:.1f}ms")


Benchmark upload/download
────────────────────────────────────────
       1 KB — upload : 2009.1ms  download : 3.5ms
      10 KB — upload : 86.7ms  download : 2.2ms
     100 KB — upload : 129.3ms  download : 4.2ms
     500 KB — upload : 136.5ms  download : 30.5ms


In [8]:
# %% [markdown]
# ## 8. Résumé

# %%
print("=" * 60)
print("  RÉSUMÉ NOTEBOOK 03 — Intégration IPFS")
print("=" * 60)
print(f"  CID texte généré   : {cid[:32]}...")
print(f"  CID JSON généré    : {cid_json[:32]}...")
print(f"  CID audio généré   : {cid_audio[:32]}...")
print(f"  Upload 100KB       : {up_times[2]*1000:.1f}ms")
print(f"  Download 100KB     : {dl_times[2]*1000:.1f}ms")
print(f"  Intégrité          : fonctionnelle")
print("=" * 60)
print("  ✅ Notebook 03 validé — passer au notebook 04")

  RÉSUMÉ NOTEBOOK 03 — Intégration IPFS
  CID texte généré   : QmajWL6tEHaw7sSaL5VaNqDjgm19qofK...
  CID JSON généré    : QmRLQVBf4b6scRw435zQdJXbRFn8wgj9...
  CID audio généré   : QmbZRF7iYNUMoQYoes51QZZ9CC5VQYhs...
  Upload 100KB       : 129.3ms
  Download 100KB     : 4.2ms
  Intégrité          : fonctionnelle
  ✅ Notebook 03 validé — passer au notebook 04
